# Pegasus Newsroom

## Introduction
This notebook demonstrates **Pegasus Newsroom** for text summarization. This pretrained checkpoint generates a domain-adapted abstractive summary.

### Objectives
- Load the shared article from `article.txt`.
- Generate a summary with this notebook's method.
- Evaluate the result against `reference_summary.txt` with shared ROUGE and BLEU metrics.

### Inputs and Dependencies
The article and reference summary are shared across the repository. Classical methods use the dependencies listed in `requirements.txt`; pretrained inference also downloads its model checkpoint.


In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("HF_HOME", str(ROOT / ".model_cache"))
os.environ.setdefault("NLTK_DATA", str(ROOT / ".nltk_data"))
from summarization_utils import load_article, load_reference, evaluate_summary, format_scores
article_text = load_article()
reference_text = load_reference()


# Pegasus Newsroom

## Introduction
This notebook demonstrates **Pegasus Newsroom** for text summarization. News article summarization.

### Objectives
- Load the shared article from `article.txt`.
- Generate a summary with this notebook's method.
- Evaluate the result against `reference_summary.txt` with shared ROUGE and BLEU metrics.

### Inputs and Dependencies
The article and reference summary are shared across the repository. Classical methods use the dependencies listed in `requirements.txt`; pretrained inference also downloads its model checkpoint.


**Pegasus-Newsroom** is a pre-trained language model introduced in 2021 for the task of summarizing news articles. It is based on the Pegasus architecture and is trained on a large and diverse dataset of news articles from various sources. The model is designed to handle multiple languages and can generate high-quality summaries that capture the most important information from the original article. Pegasus-Newsroom has achieved state-of-the-art performance on the Newsroom benchmark, which consists of news articles from multiple sources and in multiple languages. The model is available as part of the Hugging Face Transformers library, making it easy to use for natural language processing tasks.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from summarization_utils import load_article, load_reference, evaluate_summary, format_scores

article = load_article()
reference_summary = load_reference()
model_name = 'google/pegasus-newsroom'
# The MultiNews checkpoint expects document separators; all other checkpoints receive the article as one document.
model_input = article
if model_name.endswith("multi_news"):
    model_input = " ||| ".join(part.strip() for part in article.split("\n\n") if part.strip())

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
inputs = tokenizer(model_input, truncation=True, max_length=1024, return_tensors="pt")
with torch.inference_mode():
    generated = model.generate(**inputs, num_beams=4, max_length=120, min_length=20, no_repeat_ngram_size=3)
summary = tokenizer.decode(generated[0], skip_special_tokens=True)
print("Summary:", summary)
print("Metrics:", format_scores(evaluate_summary(summary, reference_summary)))
